In [32]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", None)

In [33]:
ward_type_path = "../data/closures_by_ward_and_type.csv"
biz_path       = "../data/ward_business_counts.csv"

wt = pd.read_csv(ward_type_path)
biz = pd.read_csv(biz_path)

print("Ward × type table columns:", wt.columns.tolist())
print("Business counts columns:", biz.columns.tolist())

wt.head(), biz.head()
wt.columns, biz.columns

Ward × type table columns: ['WARD', 'LICENSE DESCRIPTION', 'count']
Business counts columns: ['ward', 'biz_exposure']


(Index(['WARD', 'LICENSE DESCRIPTION', 'count'], dtype='object'),
 Index(['ward', 'biz_exposure'], dtype='object'))

In [34]:
# Correct column names for your data
WARD_COL = "WARD"
TYPE_COL = "LICENSE DESCRIPTION"
COUNT_COL = "count"
BIZ_COUNT_COL = "biz_exposure"

# Make sure biz uses the same ward column name as wt
if WARD_COL not in biz.columns:
    biz = biz.rename(columns={"ward": WARD_COL})

# (Optional safety) if the biz exposure column had a different name, we could rename it here,
# but in your case it's already 'biz_exposure', so this is just for reference:
# if BIZ_COUNT_COL not in biz.columns:
#     alt_cols = [c for c in biz.columns if c != WARD_COL]
#     assert len(alt_cols) == 1, "Please adjust BIZ_COUNT_COL manually."
#     biz = biz.rename(columns={alt_cols[0]: BIZ_COUNT_COL})

# 1) Pivot ward×type to wide: rows = ward, cols = type
wide = wt.pivot_table(
    index=WARD_COL,
    columns=TYPE_COL,
    values=COUNT_COL,
    aggfunc="sum",
    fill_value=0
)

# Make nice column names: type_xxx...
wide.columns = [f"type_{c}" for c in wide.columns]

# 2) Merge business counts
wide = wide.merge(biz[[WARD_COL, BIZ_COUNT_COL]], on=WARD_COL)

wide.head()


,WARD,type_Accessory Garage,type_Affiliation,type_Airport Pushcart Liquor Midway - Class A,type_Airport Pushcart Liquor O'Hare - Class A,type_Animal Care Facility,type_Animal Care License,type_Animal Exhibition,type_Assisted Living/Shared Housing Establishment,type_Auctioneer,type_Automatic Amusement Device Operator,type_Bed-And-Breakfast Establishment,type_Bicycle Messenger Service,type_Board-Up Work,type_Body Piercing,type_Broker,type_Caterer's Liquor License,type_Children's Activities Facilities,type_Children's Services Facility License,type_Class A - Indoor Special Event,type_Commercial Garage,type_Commercial Passenger Vessel,type_Consumption on Premises - Incidental Activity,type_Day Care Center 2 - 6 Years,type_Day Care Center Under 2 Years,type_Day Care Center Under 2 and 2 - 6 Years,type_Day Labor Agency,type_Electronic Equipment Repair,type_Emerging Business,type_Expediter - Class A,type_Expediter - Class B,type_Expediter - Class B Employee,type_Explosives,"type_Explosives, Certificate of Fitness",type_Filling Station,type_Food - Shared Kitchen,type_Food - Shared Kitchen - Supplemental,type_Grooming Facility,type_Hazardous Materials,type_Home Occupation,type_Home Repair,type_Hospital,type_Hotel,type_Humane Society,type_Indoor Special Event,type_Industrial Private Event Venue,type_Junk Peddler,type_Kennels and Catteries,type_Laboratories,type_Late Hour,"type_Laundry, Late Hour",type_License Broker,type_License Manager,type_Limited Business License,type_Liquor Airport Pushcart License,type_Long-Term Care Facility,type_Manufacturing Establishments,type_Massage Establishment,type_Massage Therapist,type_Mobile Food Dispenser,type_Mobile Food License,type_Mobile Frozen Desserts Dispenser - Non-Motorized,type_Motor Vehicle Repair : Engine Only (Class II),type_Motor Vehicle Repair: Engine/Body(Class III),type_Motor Vehicle Repair; Specialty(Class I),type_Motor Vehicle Services License,type_Music and Dance,type_Navy Pier - Mobile,type_Navy Pier - Outdoor Fixed,type_Navy Pier Kiosk License,type_Navy Pier Vendor (Food),type_Navy Pier Vendor (Non-Food),type_Night Care Privilege,type_Not-For-Profit Club,type_Outdoor Patio,type_Package Goods,type_Pawnbroker,type_Peddler License,"type_Peddler, food (fruits and vegtables only)","type_Peddler, non-food","type_Peddler, non-food, special","type_Peddler,food - (fruits and vegetables only) - special",type_Pet Shop,type_Pharmaceutical Representative,type_Pop-Up Establishment Host - Tier II,type_Pop-Up Establishment Host - Tier III,type_Pop-Up Retail User,type_Private Booting Operation,type_Private Booting Registration,type_Produce Merchant,type_Public Place of Amusement,type_Public Place of Amusement (PAV),type_Public Place of Amusement-TCC,type_Raffles,type_Regulated Business License,type_Repossessor Class A,type_Repossessor Class B Employee,type_Residential Real Estate Developer,type_Retail Computing Center,type_Retail Food Est.-Supplemental License for Dog-Friendly Areas,type_Retail Food Establishment,type_Riverwalk Venue Liquor License,"type_Scavenger, Private",type_Scooter Sharing License,type_Secondhand Dealer,type_Secondhand Dealer (No Valuable Objects),type_Secondhand Dealer - Children's Products,type_Shared Housing Unit Operator,type_Shared Kitchen User (Long Term),type_Shared Kitchen User (Short Term),type_Single Room Occupancy Class I,type_Single Room Occupancy Class II,type_Special Event Food,type_Special Event Liquor,type_Street Performer,type_Tavern,type_Taxicab Two-Way Dispatch Service License,"type_Tire Facility Class II (1,001 - 5,000 Tires)","type_Tire Facility Class III (5,001 - More Tires)","type_Tire Facilty Class I (100 - 1,000 Tires)",type_Tobacco,type_Tobacco Dealer Wholesale,type_Tobacco Sampler,"type_Tobacco Vending, Individual",type_Towing - Tow Truck,type_Transportation Network Provider,type_Vacation Rental,type_Valet Parking Operator,type_Veterinary Hospital,type_Weapons Dealer,type_Wholesale Food Establishment,type_Wrigley Field,biz_exposure
0,1.0,0,0,0,0,2,1,

In [35]:
# Compute per-1000-business rates for each type
type_cols = [c for c in wide.columns if c.startswith("type_")]

for c in type_cols:
    wide[c] = (wide[c] / wide[BIZ_COUNT_COL]) * 1000

# Total closure rate per 1000 businesses (sum of all types)
wide["total_closures_per_1000"] = wide[type_cols].sum(axis=1)

wide.head()

/var/folders/0w/lgt6rsld0y1dn8y8brbcff9r0000gn/T/ipykernel_8847/3203527908.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  wide["total_closures_per_1000"] = wide[type_cols].sum(axis=1)


,WARD,type_Accessory Garage,type_Affiliation,type_Airport Pushcart Liquor Midway - Class A,type_Airport Pushcart Liquor O'Hare - Class A,type_Animal Care Facility,type_Animal Care License,type_Animal Exhibition,type_Assisted Living/Shared Housing Establishment,type_Auctioneer,type_Automatic Amusement Device Operator,type_Bed-And-Breakfast Establishment,type_Bicycle Messenger Service,type_Board-Up Work,type_Body Piercing,type_Broker,type_Caterer's Liquor License,type_Children's Activities Facilities,type_Children's Services Facility License,type_Class A - Indoor Special Event,type_Commercial Garage,type_Commercial Passenger Vessel,type_Consumption on Premises - Incidental Activity,type_Day Care Center 2 - 6 Years,type_Day Care Center Under 2 Years,type_Day Care Center Under 2 and 2 - 6 Years,type_Day Labor Agency,type_Electronic Equipment Repair,type_Emerging Business,type_Expediter - Class A,type_Expediter - Class B,type_Expediter - Class B Employee,type_Explosives,"type_Explosives, Certificate of Fitness",type_Filling Station,type_Food - Shared Kitchen,type_Food - Shared Kitchen - Supplemental,type_Grooming Facility,type_Hazardous Materials,type_Home Occupation,type_Home Repair,type_Hospital,type_Hotel,type_Humane Society,type_Indoor Special Event,type_Industrial Private Event Venue,type_Junk Peddler,type_Kennels and Catteries,type_Laboratories,type_Late Hour,"type_Laundry, Late Hour",type_License Broker,type_License Manager,type_Limited Business License,type_Liquor Airport Pushcart License,type_Long-Term Care Facility,type_Manufacturing Establishments,type_Massage Establishment,type_Massage Therapist,type_Mobile Food Dispenser,type_Mobile Food License,type_Mobile Frozen Desserts Dispenser - Non-Motorized,type_Motor Vehicle Repair : Engine Only (Class II),type_Motor Vehicle Repair: Engine/Body(Class III),type_Motor Vehicle Repair; Specialty(Class I),type_Motor Vehicle Services License,type_Music and Dance,type_Navy Pier - Mobile,type_Navy Pier - Outdoor Fixed,type_Navy Pier Kiosk License,type_Navy Pier Vendor (Food),type_Navy Pier Vendor (Non-Food),type_Night Care Privilege,type_Not-For-Profit Club,type_Outdoor Patio,type_Package Goods,type_Pawnbroker,type_Peddler License,"type_Peddler, food (fruits and vegtables only)","type_Peddler, non-food","type_Peddler, non-food, special","type_Peddler,food - (fruits and vegetables only) - special",type_Pet Shop,type_Pharmaceutical Representative,type_Pop-Up Establishment Host - Tier II,type_Pop-Up Establishment Host - Tier III,type_Pop-Up Retail User,type_Private Booting Operation,type_Private Booting Registration,type_Produce Merchant,type_Public Place of Amusement,type_Public Place of Amusement (PAV),type_Public Place of Amusement-TCC,type_Raffles,type_Regulated Business License,type_Repossessor Class A,type_Repossessor Class B Employee,type_Residential Real Estate Developer,type_Retail Computing Center,type_Retail Food Est.-Supplemental License for Dog-Friendly Areas,type_Retail Food Establishment,type_Riverwalk Venue Liquor License,"type_Scavenger, Private",type_Scooter Sharing License,type_Secondhand Dealer,type_Secondhand Dealer (No Valuable Objects),type_Secondhand Dealer - Children's Products,type_Shared Housing Unit Operator,type_Shared Kitchen User (Long Term),type_Shared Kitchen User (Short Term),type_Single Room Occupancy Class I,type_Single Room Occupancy Class II,type_Special Event Food,type_Special Event Liquor,type_Street Performer,type_Tavern,type_Taxicab Two-Way Dispatch Service License,"type_Tire Facility Class II (1,001 - 5,000 Tires)","type_Tire Facility Class III (5,001 - More Tires)","type_Tire Facilty Class I (100 - 1,000 Tires)",type_Tobacco,type_Tobacco Dealer Wholesale,type_Tobacco Sampler,"type_Tobacco Vending, Individual",type_Towing - Tow Truck,type_Transportation Network Provider,type_Vacation Rental,type_Valet Parking Operator,type_Veterinary Hospital,type_Weapons Dealer,type_Wholesale Food Establishment,type_Wrigley Field,biz_exposure,total_closures_per

In [36]:
# Features we will use for the embedding
feat_cols = type_cols + ["total_closures_per_1000"]

scaler = StandardScaler()
wide_scaled = wide.copy()
wide_scaled[feat_cols] = scaler.fit_transform(wide[feat_cols])

# Reset index so 'ward' becomes a column again
embeddings = wide_scaled.reset_index()  # ward column comes back

embeddings.head()

,index,WARD,type_Accessory Garage,type_Affiliation,type_Airport Pushcart Liquor Midway - Class A,type_Airport Pushcart Liquor O'Hare - Class A,type_Animal Care Facility,type_Animal Care License,type_Animal Exhibition,type_Assisted Living/Shared Housing Establishment,type_Auctioneer,type_Automatic Amusement Device Operator,type_Bed-And-Breakfast Establishment,type_Bicycle Messenger Service,type_Board-Up Work,type_Body Piercing,type_Broker,type_Caterer's Liquor License,type_Children's Activities Facilities,type_Children's Services Facility License,type_Class A - Indoor Special Event,type_Commercial Garage,type_Commercial Passenger Vessel,type_Consumption on Premises - Incidental Activity,type_Day Care Center 2 - 6 Years,type_Day Care Center Under 2 Years,type_Day Care Center Under 2 and 2 - 6 Years,type_Day Labor Agency,type_Electronic Equipment Repair,type_Emerging Business,type_Expediter - Class A,type_Expediter - Class B,type_Expediter - Class B Employee,type_Explosives,"type_Explosives, Certificate of Fitness",type_Filling Station,type_Food - Shared Kitchen,type_Food - Shared Kitchen - Supplemental,type_Grooming Facility,type_Hazardous Materials,type_Home Occupation,type_Home Repair,type_Hospital,type_Hotel,type_Humane Society,type_Indoor Special Event,type_Industrial Private Event Venue,type_Junk Peddler,type_Kennels and Catteries,type_Laboratories,type_Late Hour,"type_Laundry, Late Hour",type_License Broker,type_License Manager,type_Limited Business License,type_Liquor Airport Pushcart License,type_Long-Term Care Facility,type_Manufacturing Establishments,type_Massage Establishment,type_Massage Therapist,type_Mobile Food Dispenser,type_Mobile Food License,type_Mobile Frozen Desserts Dispenser - Non-Motorized,type_Motor Vehicle Repair : Engine Only (Class II),type_Motor Vehicle Repair: Engine/Body(Class III),type_Motor Vehicle Repair; Specialty(Class I),type_Motor Vehicle Services License,type_Music and Dance,type_Navy Pier - Mobile,type_Navy Pier - Outdoor Fixed,type_Navy Pier Kiosk License,type_Navy Pier Vendor (Food),type_Navy Pier Vendor (Non-Food),type_Night Care Privilege,type_Not-For-Profit Club,type_Outdoor Patio,type_Package Goods,type_Pawnbroker,type_Peddler License,"type_Peddler, food (fruits and vegtables only)","type_Peddler, non-food","type_Peddler, non-food, special","type_Peddler,food - (fruits and vegetables only) - special",type_Pet Shop,type_Pharmaceutical Representative,type_Pop-Up Establishment Host - Tier II,type_Pop-Up Establishment Host - Tier III,type_Pop-Up Retail User,type_Private Booting Operation,type_Private Booting Registration,type_Produce Merchant,type_Public Place of Amusement,type_Public Place of Amusement (PAV),type_Public Place of Amusement-TCC,type_Raffles,type_Regulated Business License,type_Repossessor Class A,type_Repossessor Class B Employee,type_Residential Real Estate Developer,type_Retail Computing Center,type_Retail Food Est.-Supplemental License for Dog-Friendly Areas,type_Retail Food Establishment,type_Riverwalk Venue Liquor License,"type_Scavenger, Private",type_Scooter Sharing License,type_Secondhand Dealer,type_Secondhand Dealer (No Valuable Objects),type_Secondhand Dealer - Children's Products,type_Shared Housing Unit Operator,type_Shared Kitchen User (Long Term),type_Shared Kitchen User (Short Term),type_Single Room Occupancy Class I,type_Single Room Occupancy Class II,type_Special Event Food,type_Special Event Liquor,type_Street Performer,type_Tavern,type_Taxicab Two-Way Dispatch Service License,"type_Tire Facility Class II (1,001 - 5,000 Tires)","type_Tire Facility Class III (5,001 - More Tires)","type_Tire Facilty Class I (100 - 1,000 Tires)",type_Tobacco,type_Tobacco Dealer Wholesale,type_Tobacco Sampler,"type_Tobacco Vending, Individual",type_Towing - Tow Truck,type_Transportation Network Provider,type_Vacation Rental,type_Valet Parking Operator,type_Veterinary Hospital,type_Weapons Dealer,type_Wholesale Food Establishment,type_Wrigley Field,biz_exposure,total_closur

In [37]:
# Save embeddings (one row per ward, columns = ward + features)
out_path = "../web/embeddings.csv"
embeddings.to_csv(out_path, index=False)
print("Saved embeddings to", out_path)

Saved embeddings to ../web/embeddings.csv


In [38]:
# PCA to 2D on the feature columns
pca = PCA(n_components=2, random_state=0)
coords = pca.fit_transform(embeddings[feat_cols])

emb_2d = embeddings[[WARD_COL]].copy()
emb_2d["x"] = coords[:, 0]
emb_2d["y"] = coords[:, 1]

emb_2d.head()

,WARD,x,y
0,1.0,3.996051,-1.299425
1,2.0,10.373307,0.387088
2,3.0,2.642656,0.460496
3,4.0,4.044714,-0.132154
4,5.0,2.553062,0.317371


In [39]:
out_2d_path = "../web/embeddings_2d.csv"
emb_2d.to_csv(out_2d_path, index=False)
print("Saved 2D projection to", out_2d_path)

Saved 2D projection to ../web/embeddings_2d.csv
